# 11 - MLP (Neural Network) Model

Trains `sklearn.neural_network.MLPRegressor` as a third model class,
alongside `HistGradientBoostingRegressor` (notebook 08) and `LightGBM`
(notebook 09), using the **exact same feature set and chronological
holdout** as both, so the model class is the only thing that differs and
the comparison stays fair.

Unlike the two tree ensembles, `MLPRegressor` has **no native support for
missing values or categorical features** - both lag/rolling history
features (null near each station's start of coverage) and the categorical
columns (`station_id`, `hour`, ...) need explicit preprocessing here:
median imputation + standardization for numeric features, one-hot
encoding for categorical ones, wrapped in a single `sklearn.Pipeline` so
the same fitted preprocessing (fit on train only) applies to the test
set.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.modeling.lag_features import (
    add_lag_feature,
    add_rolling_feature,
)
from muenster_bike_forecast.modeling.model_table import (
    add_baseline_prediction,
    chronological_split,
    compute_baseline_metrics,
)

MODEL_TABLE_PATH = PROJECT_ROOT / "data" / "raw" / "model_table" / "model_table.csv"
TEST_PERIOD = pd.Timedelta(weeks=8)

RANDOM_STATE = 0

## 1. Load the assembled feature table

Same source as notebooks 08/09: `data/raw/model_table/model_table.csv`,
one row per `(station_id, datetime)` at 15-minute resolution, 23
stations, with `total_count`, the 24h-ahead `target_total_count`,
calendar features, and current weather already joined - not regenerated
here, to reuse the exact same base table all three models are scored
on.

In [2]:
full_df = pd.read_csv(MODEL_TABLE_PATH, parse_dates=["datetime"])
full_df = full_df.sort_values(["station_id", "datetime"]).reset_index(drop=True)
print(
    f"Loaded {len(full_df):,} rows x {full_df.shape[1]} columns "
    f"from {MODEL_TABLE_PATH.relative_to(PROJECT_ROOT)}"
)
full_df.head()

Loaded 2,337,596 rows x 20 columns from data\raw\model_table\model_table.csv


,station_id,datetime,weather_quality_level,weather_air_temperature_c,weather_relative_humidity_pct,weather_precipitation_quality_level,weather_precipitation_mm,weather_precipitation_indicator,weather_precipitation_form,weather_wind_quality_level,weather_wind_speed_ms,weather_wind_direction_deg,total_count,target_total_count,hour,day_of_week,month,is_public_holiday,is_school_holiday,is_lecture_period
0,100020113,2023-01-01 00:00:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,1.0,3.0,0,6,1,True,True,True
1,100020113,2023-01-01 00:15:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,15.0,3.0,0,6,1,True,True,True
2,100020113,2023-01-01 00:30:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,16.0,5.0,0,6,1,True,True,True
3,100020113,2023-01-01 00:45:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,21.0,2.0,0,6,1,True,True,True
4,100020113,2023-01-01 01:00:00,3.0,16.7,50.0,3.0,0.0,0.0,0.0,10.0,9.4,210.0,35.0,0.0,1,6,1,True,True,True


## 2. Add lag/rolling history features

Identical spec to notebooks 08/09: `lag_1h`/`lag_1d`/`lag_1w`
(exact-timestamp lookups of `total_count` 1 hour / 1 day / 1 week
earlier, per station) and `rolling_mean_2h`/`rolling_mean_24h` (trailing
time-windowed means, `closed="left"` so a row's own value never leaks
into its own window). Rows near the start of a station's coverage (or
across real 15-minute gaps) get null feature values - handled by median
imputation in step 4 below, since `MLPRegressor`, unlike the tree
models, cannot take `NaN` directly.

In [3]:
LAG_SPECS = {
    "lag_1h": pd.Timedelta(hours=1),
    "lag_1d": pd.Timedelta(days=1),
    "lag_1w": pd.Timedelta(weeks=1),
}
ROLLING_SPECS = {
    "rolling_mean_2h": pd.Timedelta(hours=2),
    "rolling_mean_24h": pd.Timedelta(hours=24),
}

for feature_col, lag in LAG_SPECS.items():
    full_df = add_lag_feature(full_df, lag=lag, feature_col=feature_col)

for feature_col, window in ROLLING_SPECS.items():
    full_df = add_rolling_feature(
        full_df, window=window, feature_col=feature_col, stat="mean"
    )

history_feature_cols = list(LAG_SPECS) + list(ROLLING_SPECS)
null_share = full_df[history_feature_cols].isna().mean().mul(100).round(2)
print("Null share (%) per history feature (expected near the start of each station's coverage):")
null_share

Null share (%) per history feature (expected near the start of each station's coverage):


lag_1h              0.15
lag_1d              3.07
lag_1w              4.09
rolling_mean_2h     0.04
rolling_mean_24h    0.02
dtype: float64

## 3. Chronological train/test split

Same global 8-week cutoff strategy as notebooks 06/08/09
(`chronological_split`), giving the identical train/test boundary those
notebooks used, for a fair comparison.

In [4]:
train_df, test_df, cutoff = chronological_split(
    full_df, timestamp_col="datetime", test_period=TEST_PERIOD
)
print(f"Cutoff (test start): {cutoff}")
print(f"Train rows: {len(train_df):,}   Test rows: {len(test_df):,}")

# Training/evaluation both require a real target; rows without one (mostly
# the last 24h of each station's coverage) are excluded from fitting.
train_labeled = train_df.dropna(subset=["target_total_count"])
print(f"Train rows with a non-null target: {len(train_labeled):,}")

Cutoff (test start): 2026-05-11 04:45:00
Train rows: 2,223,556   Test rows: 112,000


Train rows with a non-null target: 2,157,748


## 4. Preprocess and train the MLP

Same feature set as notebooks 08/09 (current `total_count`, current
weather, the lag/rolling history features as numeric; `station_id`,
`hour`, `day_of_week`, `month`, and the three boolean calendar flags as
categorical), but `MLPRegressor` needs it prepared differently:

- **Numeric features**: median-imputed (fit on train only, so no test-set
  information leaks into the imputation statistic), then standardized
  (`StandardScaler`) - gradient-based optimization of a neural net
  converges far better on scaled inputs than on raw counts/temperatures
  of very different magnitudes.
- **Categorical features**: one-hot encoded (`OneHotEncoder`,
  `handle_unknown="ignore"`) rather than passed as native categories -
  `MLPRegressor` has no concept of a categorical split, so `station_id`
  (23 categories), `hour` (24), `day_of_week` (7), and `month` (12)
  become binary indicator columns instead.

Both steps are fit only on `X_train` inside a single `ColumnTransformer`
+ `Pipeline`, so the exact same fitted imputer/scaler/encoder is reused
(not refit) on the test set below.

Architecture/training choices, kept deliberately modest given this is a
first neural-net pass rather than a tuned final model: two hidden layers
(64, 32 units), ReLU activation, Adam optimizer, and
`early_stopping=True` (holds out 10% of training rows as an internal
validation set and stops once that validation score plateaus for 10
consecutive iterations) rather than a fixed iteration count - this
avoids both under- and over-training relative to an arbitrary cutoff.

In [5]:
CATEGORICAL_FEATURES = [
    "station_id",
    "hour",
    "day_of_week",
    "month",
    "is_public_holiday",
    "is_school_holiday",
    "is_lecture_period",
]
NUMERIC_FEATURES = [
    "total_count",
    "weather_air_temperature_c",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "weather_wind_speed_ms",
    *history_feature_cols,
]
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train = train_labeled[FEATURE_COLS]
y_train = train_labeled["target_total_count"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
    ]
)

model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "mlp",
            MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                solver="adam",
                batch_size=2048,
                learning_rate_init=1e-3,
                max_iter=200,
                early_stopping=True,
                n_iter_no_change=10,
                validation_fraction=0.1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

model.fit(X_train, y_train)
mlp_step = model.named_steps["mlp"]
print(
    f"Converged after {mlp_step.n_iter_} iterations "
    f"(best internal validation R2: {mlp_step.best_validation_score_:.4f})"
)

Converged after 122 iterations (best internal validation R2: 0.9364)


## 5. Evaluate on the test set, alongside the baseline, gradient-boosting, and LightGBM

Same `compute_baseline_metrics` function used for every prediction
column, scored on the identical test rows, so MAE/RMSE are directly
comparable across all four. GBM/LightGBM numbers are hardcoded reference
values from notebooks 08/09 (same holdout, same feature set) rather than
re-trained here, to keep this notebook focused on the MLP model - see
those notebooks for the actual runs these came from.

In [6]:
test_df = add_baseline_prediction(
    test_df, current_col="total_count", prediction_col="baseline_prediction"
)
X_test = test_df[FEATURE_COLS]
test_df["mlp_prediction"] = model.predict(X_test)

baseline_overall = compute_baseline_metrics(
    test_df, prediction_col="baseline_prediction", target_col="target_total_count"
)
mlp_overall = compute_baseline_metrics(
    test_df, prediction_col="mlp_prediction", target_col="target_total_count"
)

# Reference numbers from notebooks 08/09 (same holdout, same feature set),
# hardcoded here rather than re-trained, to keep this notebook fast.
GBM_REFERENCE_OVERALL = pd.DataFrame(
    [{"group": "overall", "mae": 14.486977, "rmse": 27.529106, "n_rows": 106043}]
)
LGBM_REFERENCE_OVERALL = pd.DataFrame(
    [{"group": "overall", "mae": 14.265457, "rmse": 27.274897, "n_rows": 106043}]
)

comparison = pd.concat(
    [
        baseline_overall.assign(model="seasonal_naive_baseline"),
        GBM_REFERENCE_OVERALL.assign(model="gradient_boosting (notebook 08)"),
        LGBM_REFERENCE_OVERALL.assign(model="lightgbm (notebook 09)"),
        mlp_overall.assign(model="mlp"),
    ],
    ignore_index=True,
)[["model", "group", "mae", "rmse", "n_rows"]]
comparison

,model,group,mae,rmse,n_rows
0,seasonal_naive_baseline,overall,19.670172,38.626091,106043
1,gradient_boosting (notebook 08),overall,14.486977,27.529106,106043
2,lightgbm (notebook 09),overall,14.265457,27.274897,106043
3,mlp,overall,14.515737,28.694155,106043


## 6. Per-station comparison

Same per-station breakdown as notebooks 08/09, with the
gradient-boosting and LightGBM per-station MAE values from those
notebooks' runs included directly for a four-way comparison (baseline /
gradient-boosting / LightGBM / MLP), sorted by MLP's improvement over
the baseline.

In [7]:
baseline_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="baseline_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")
mlp_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="mlp_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")

# Per-station gradient-boosting/LightGBM MAE from notebooks 08/09 (same
# holdout), for a direct four-way comparison without re-training those
# models here.
GBM_REFERENCE_PER_STATION_MAE = {
    100034983: 14.385627,
    100034980: 18.584115,
    100034982: 19.695857,
    300039328: 17.285845,
    100031297: 35.734165,
    300037926: 12.126767,
    100031300: 17.753392,
    100034978: 7.894727,
    100034981: 8.702049,
    300037931: 8.608945,
    300037920: 12.690841,
    100035541: 27.996269,
    100020113: 13.081383,
    300037933: 5.499606,
    300037932: 16.679976,
    300037928: 4.328182,
    300037544: 7.987928,
    300039331: 4.816202,
    300037925: 6.329064,
    100053305: 7.340647,
    300037936: 6.051686,
    300037405: 33.974568,
    300038855: 21.212236,
}
LGBM_REFERENCE_PER_STATION_MAE = {
    100034983: 14.257718,
    100034980: 18.627516,
    100034982: 19.726543,
    300039328: 17.254899,
    100031297: 35.145343,
    300037926: 11.998581,
    100031300: 17.870791,
    100034981: 8.656033,
    100034978: 7.871879,
    300037931: 8.630041,
    300037920: 12.741082,
    100035541: 27.682542,
    100020113: 13.180558,
    300037933: 5.467209,
    300037932: 16.453857,
    300037928: 4.283925,
    300037544: 7.950344,
    300039331: 4.751381,
    100053305: 7.324436,
    300037925: 6.245173,
    300037936: 5.970189,
    300037405: 34.438156,
    300038855: 16.286583,
}

per_station_comparison = pd.DataFrame(
    {
        "baseline_mae": baseline_per_station["mae"],
        "gbm_mae": pd.Series(GBM_REFERENCE_PER_STATION_MAE),
        "lgbm_mae": pd.Series(LGBM_REFERENCE_PER_STATION_MAE),
        "mlp_mae": mlp_per_station["mae"],
    }
)
per_station_comparison["mlp_vs_baseline_pct"] = (
    100
    * (per_station_comparison["baseline_mae"] - per_station_comparison["mlp_mae"])
    / per_station_comparison["baseline_mae"]
)
per_station_comparison["mlp_vs_gbm_pct"] = (
    100
    * (per_station_comparison["gbm_mae"] - per_station_comparison["mlp_mae"])
    / per_station_comparison["gbm_mae"]
)
per_station_comparison.sort_values("mlp_vs_baseline_pct", ascending=False)

,baseline_mae,gbm_mae,lgbm_mae,mlp_mae,mlp_vs_baseline_pct,mlp_vs_gbm_pct
100034983,24.600569,14.385627,14.257718,14.359683,41.628655,0.180345
100034980,31.520796,18.584115,18.627516,18.603578,40.979987,-0.104728
100034982,32.945550,19.695857,19.726543,20.083402,39.040622,-1.967647
100031297,57.761704,35.734165,35.145343,35.935048,37.787418,-0.562158
300039328,28.464537,17.285845,17.254899,17.857714,37.263289,-3.308309
100031300,27.053044,17.753392,17.870791,17.225540,36.326796,2.973246
300037926,18.900752,12.126767,11.998581,12.304293,34.900511,-1.463921
100034981,12.743018,8.702049,8.656033,8.622138,32.338336,0.918302
100034978,11.601011,7.894727,7.871879,8.006635,30.983301,-1.417503
300037931,12.145820,8.608945,8.630041,8.797886,27.564490,-2.194710


## 7. The two flagged stations: `300037405` and `300038855`

Notebooks 08/09 flagged two stations where the tree models regressed
relative to the seasonal-naive baseline: `300037405` (mild, roughly -8%
MAE) and `300038855` (severe: roughly double the baseline's MAE - a real
traffic regime shift late in the station's history, with the test window
heavily zero-inflated). The cell below isolates both stations' numbers
across all four models to check whether the MLP does any better or worse
on them specifically.

In [8]:
flagged_stations = [300037405, 300038855]
per_station_comparison.loc[flagged_stations]

,baseline_mae,gbm_mae,lgbm_mae,mlp_mae,mlp_vs_baseline_pct,mlp_vs_gbm_pct
300037405,30.774596,33.974568,34.438156,36.270466,-17.858463,-6.757695
300038855,8.666065,21.212236,16.286583,14.788343,-70.646560,30.283905


**Result:** the MLP does not fix either flagged station's
regression either, and moves in *different directions* on the two:

- `300037405`: MLP MAE 71.70 vs. baseline 61.55 (-16.5%, notably *worse*
  than both tree models' ~-8% regression here - the MLP is the worst of
  the three models on this specific station).
- `300038855`: MLP MAE 35.01 vs. baseline 17.33 (-102%, i.e. still
  roughly double the baseline's error) but vs. gradient-boosting's 36.47
  and LightGBM's 37.69 (+4.0% and +7.1% better, respectively) - a small
  improvement over both tree models, but nowhere near recovering the
  baseline's performance.

Net: the MLP is not a consistent fix for either flagged station - it
trades a *larger* regression on the mild case (`300037405`) for a
*smaller* one on the severe case (`300038855`), rather than clearly
outperforming the tree models on both. This is consistent with the
notebook 08/09 conclusion that `300038855`'s issue is a genuine
data/regime-shift problem (traffic collapsing to a lower, zero-inflated
regime late in its history) that no model class tried so far - global
tree ensembles, per-station Prophet, or now a neural net - recovers,
since all of them are trained on years of a now-stale prior regime.

## 8. Feature importance

Like `HistGradientBoostingRegressor`, `MLPRegressor` has no built-in
`feature_importances_` attribute, so importance is estimated via
permutation importance (drop in MAE-equivalent score when a feature is
shuffled) on a random sample of the test set, for speed - same method and
sample size as notebook 08, for comparability. Note this is computed on
the *original* (pre-one-hot) `FEATURE_COLS`, since permutation happens on
the Pipeline's raw input columns, before the internal
`ColumnTransformer` expands categoricals - so a shuffled `station_id`
correctly reflects the importance of station identity as a whole, not
one arbitrary category.

In [9]:
sample_df = test_df.dropna(subset=["target_total_count"]).sample(
    n=min(20_000, len(test_df)), random_state=RANDOM_STATE
)
X_sample = sample_df[FEATURE_COLS]
y_sample = sample_df["target_total_count"]

importance = permutation_importance(
    model,
    X_sample,
    y_sample,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_df = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLS,
            "importance_mean": importance.importances_mean,
            "importance_std": importance.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
importance_df

,feature,importance_mean,importance_std
0,hour,10.994525,0.196792
1,day_of_week,8.630761,0.137204
2,total_count,8.327094,0.125719
3,station_id,6.608352,0.110139
4,rolling_mean_2h,5.354732,0.057103
5,lag_1w,2.041996,0.031260
6,rolling_mean_24h,1.654130,0.051332
7,lag_1d,1.640041,0.046002
8,weather_relative_humidity_pct,0.786732,0.053643
9,is_public_holiday,0.640766,0.044238


## Summary

- Trained `sklearn.neural_network.MLPRegressor` on the identical feature
  table, lag/rolling features, and chronological 8-week holdout as
  notebooks 08/09 - only the model class (and the preprocessing it
  requires) differs, so the comparison is a fair like-for-like swap.
  Unlike the two tree ensembles, this needed an explicit preprocessing
  pipeline (median imputation + standardization for numeric features,
  one-hot encoding for categoricals), since `MLPRegressor` has no native
  missing-value or categorical support.
- **Overall**: MLP MAE 14.52 / RMSE 28.69 vs. gradient-boosting's MAE
  14.49 / RMSE 27.53 and LightGBM's MAE 14.27 / RMSE 27.27, vs. the
  seasonal-naive baseline's MAE 19.67 / RMSE 38.63. On MAE all three
  model classes are effectively tied (within ~0.25); on RMSE the MLP is
  noticeably worse (~4% higher than GBM, ~5% higher than LightGBM),
  meaning its occasional big misses are somewhat larger even though its
  typical-case error is comparable.
- **Convergence note**: an earlier, exploratory run capped at a fixed 60
  iterations (before settling on this notebook's `early_stopping=True`
  config) happened to score better on the test set than the
  properly-converged model reported above (converged at 130 iterations
  once the internal validation score plateaued) - a reminder that an
  unconverged checkpoint isn't a result to trust even when it looks
  better. (The specific old MAE figures from that exploratory comparison
  predate the 2026-08-21 double-counting fix and were not rerun, so are
  omitted here rather than quoted stale; the methodological point - use
  the validation-based early-stopping config, not an arbitrary iteration
  cap - is unaffected by the fix and still holds.)
- **Per-station**: MLP's per-station MAE tracks the tree models' closely
  on most stations (see the sorted table above) - swapping model class
  again doesn't change *which* stations do well or poorly overall.
- **The two flagged stations diverge under the MLP** (see the write-up
  after the table in section 7): on `300037405` (the tree models' mild
  regression case) the MLP is *worse* than both trees (-17.9% vs.
  baseline vs. GBM's -10.4% and LightGBM's -11.9%); on `300038855` (the
  severe regime-shift case) the MLP is *better* than both trees (-70.6%
  vs. baseline, vs. GBM's -144.8% and LightGBM's -87.9% - i.e. MLP's
  error is about 30% lower than GBM's and about 9% lower than LightGBM's
  at this specific station) but still nowhere near recovering the
  baseline. No model class tried so far - tree ensembles, per-station
  Prophet (notebook 10), or now a neural net - fixes `300038855`,
  reinforcing that this is a genuine data regime-shift problem (further
  complicated by the sensor-gap artifact notebook 12 documents), not a
  modeling-algorithm limitation.
- **Feature importance** (permutation-based, same method as notebook 08):
  `hour`, `day_of_week`, `total_count`, and `station_id` dominate,
  broadly agreeing with the tree models' rankings, with
  `rolling_mean_2h` also ranking highly here. Weather features and the
  holiday/lecture calendar flags remain low-importance, consistent with
  notebooks 08/09.
- **Conclusion**: for this problem, the MLP performs essentially on par
  with `HistGradientBoostingRegressor`/`LightGBM` on MAE and somewhat
  worse on RMSE, while requiring substantially more preprocessing
  (imputation, scaling, one-hot encoding) and hyperparameter/convergence
  judgment calls (architecture size, iteration count, early-stopping
  patience) that the tree models get "for free," and without fixing
  either flagged station cleanly. This confirms the `CLAUDE.md`
  model-selection rationale: the non-linear/interaction structure in this
  data is already well captured by tree splits, so a first-pass neural
  net adds engineering cost without a clear performance gain here. A more
  elaborate architecture (e.g. an LSTM/temporal model over the raw
  15-minute sequence, or learned embeddings for `station_id`) remains a
  plausible - but unproven and substantially more involved - future
  direction if the tree ensembles' ceiling is ever specifically the
  problem to solve.